# 🎬 Reverse Prompt Engineering for AI Educational Videos
**向上溯源反推系统 - Google Drive 实时同步版**

本 Notebook 直接从 Google Drive 读取项目代码（无需 git clone）。
在本地修改代码后，Drive 自动同步，Colab 重新运行即可获得最新版本。

**Pipeline**: Mount Drive → Install Deps → Download Videos → Run Pipeline → Pattern Analysis

In [ ]:
# Cell 1: Check GPU
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
vram = f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else 'N/A'
print(f'GPU : {gpu_name}')
print(f'VRAM: {vram}')
if not torch.cuda.is_available():
    print('⚠️  Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive & Set Project Path
from google.colab import drive
import os

drive.mount('/content/drive')

# ⚙️ 修改为你的项目在 Drive 中的路径（相对于 MyDrive）
PROJECT_SUBPATH = '工坊省级项目'
PROJECT_DIR = f'/content/drive/MyDrive/{PROJECT_SUBPATH}'

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f'❌ 项目目录不存在: {PROJECT_DIR}\n请确认 PROJECT_SUBPATH 路径设置正确')

os.chdir(PROJECT_DIR)
print(f'✅ Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

In [ ]:
# Cell 3: Install Dependencies
!pip install -r requirements.txt -q
print('✅ Dependencies installed')

In [ ]:
# Cell 4: Cache HuggingFace Models to Google Drive
# 首次运行下载 InternVL2-2B(~4GB) + Qwen2.5-1.5B(~3GB)，缓存到 Drive 后后续秒加载
import os
HF_CACHE_DIR = '/content/drive/MyDrive/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_DIR
print(f'✅ HuggingFace 模型缓存: {HF_CACHE_DIR}')
print('   首次运行自动下载模型权重到 Drive (~7GB)，后续 session 直接从 Drive 加载')

In [ ]:
# Cell 5: Download Video Dataset (Bilibili)
# Golden: 高质量动画教学视频 (3Blue1Brown, Kurzgesagt 系列)
# Regular: 普通教学视频 (PPT/白板/录屏)
# 视频存在 Drive 里，永久保存，下次跳过
import os
MAX_VIDEOS = 5

golden_dir = 'data/raw_videos/golden'
regular_dir = 'data/raw_videos/regular'
g_count = len([f for f in os.listdir(golden_dir) if f.endswith('.mp4')]) if os.path.exists(golden_dir) else 0
r_count = len([f for f in os.listdir(regular_dir) if f.endswith('.mp4')]) if os.path.exists(regular_dir) else 0
print(f'已有视频: Golden={g_count}, Regular={r_count}')

if g_count < MAX_VIDEOS or r_count < MAX_VIDEOS:
    !python main.py --download --max {MAX_VIDEOS}
else:
    print('✅ 视频已存在，跳过下载')

In [ ]:
# Cell 6: Run Pipeline - Golden Videos
# 关键帧提取 → InternVL2-2B视觉分析 → 3个Agent推理 → 提示词合成
!python main.py --batch data/raw_videos/golden/ --group golden --output results/golden/ --lang zh

In [ ]:
# Cell 7: Run Pipeline - Regular Videos
!python main.py --batch data/raw_videos/regular/ --group regular --output results/regular/ --lang zh

In [ ]:
# Cell 8: Pattern Analysis (CPU only)
import os, shutil
os.makedirs('results/all', exist_ok=True)
for src_dir in ['results/golden', 'results/regular']:
    if os.path.exists(src_dir):
        for f in os.listdir(src_dir):
            if f.endswith('.json'):
                shutil.copy(os.path.join(src_dir, f), 'results/all/')
!python main.py --analyze --results results/all/
print('✅ 分析完成，结果已保存到 Drive')

In [ ]:
# Cell 9: View Results
import json, os
report_path = 'results/all/pattern_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    print('=== 🏆 Golden 高频关键词 ===')
    print(', '.join(report.get('golden_keywords', [])[:15]))
    print('\n=== 📚 Regular 高频关键词 ===')
    print(', '.join(report.get('regular_keywords', [])[:15]))
    print('\n=== 📊 统计 ===')
    print(json.dumps(report.get('stats', {}), indent=2))
    print('\n=== 💡 摘要 ===')
    print(report.get('summary', ''))
    print('\n=== 🥇 关键词优势对比 (Top 10) ===')
    for item in report.get('keyword_frequency', [])[:10]:
        bar = '█' * max(0, int(item['golden_advantage'] * 10))
        print(f"  {item['keyword']:20s} G:{item['golden_pct']:5.1f}% R:{item['regular_pct']:5.1f}% {bar}")
else:
    print('❌ 未找到报告，请先运行 Cell 8')

In [ ]:
# Cell 10: GPU Memory Check
import torch
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    resv  = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory: {alloc:.1f}GB used / {resv:.1f}GB reserved / {total:.1f}GB total')
    print(f'Available : {total - resv:.1f}GB')